In [ ]:
# =============================================================================
# Nine18 ARC-AGI-3 exact-agents actual scored run launcher
#
# Purpose:
#   * Run ONLY as an official Kaggle competition rerun.
#   * Use the exact bundled benchmark solver / agent objects from the TAAF bundle.
#   * Do not create offline dummy submissions.
#   * Do not substitute fallback, stub, mock, random-placeholder, or notebook-local agents.
#   * Let the bundled teardown create the real submission.parquet.
# =============================================================================

import inspect
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()
REQUIRE_TRUE_SUBMISSION = os.environ.get("NINE18_REQUIRE_TRUE_SUBMISSION", "1").strip().lower() not in {"0", "false", "no", "off"}

print(f"nine18.exact: TRUE_SUBMISSION={TRUE_SUBMISSION} REQUIRE_TRUE_SUBMISSION={REQUIRE_TRUE_SUBMISSION}")
if REQUIRE_TRUE_SUBMISSION and not TRUE_SUBMISSION:
    raise RuntimeError(
        "This notebook is configured for an actual scored ARC-AGI-3 run only. "
        "Submit it through Kaggle's official competition rerun path so "
        "KAGGLE_IS_COMPETITION_RERUN=true. No offline dummy submission will be produced."
    )

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# The benchmark and solver should behave as a real submission in the official rerun.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_EXACT_AGENTS_ONLY"] = "1"
os.environ["NINE18_EXACT_AGENTS_ONLY"] = "1"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"nine18.exact: working dir = {WORKING_DIR}")

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip().split("\n")[0]
except Exception as exc:
    gpu_name = f"<nvidia-smi failed: {exc}>"
print(f"nine18.exact: attached GPU = {gpu_name}")
if TRUE_SUBMISSION and "H100" not in gpu_name:
    print(
        f"nine18.exact: WARNING - attached GPU is '{gpu_name}'. "
        "If an attached wheelhouse was built for a different GPU architecture, setup may fail."
    )

# =============================================================================
# 2. Install the ARC runtime from the offline competition wheelhouse
# =============================================================================

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

# =============================================================================
# 3. Locate the bundled exact-agent source and attached Kaggle inputs
# =============================================================================

DATASET_SOURCES = [
    "jeroencottaar/taaf-kaggle-source-share",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
    "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot",
]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"nine18.exact: source bundle = {BUNDLE_DIR}")

kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"nine18.exact: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

# =============================================================================
# 4. Import the bundled source and run the bundle's exact setup commands
# =============================================================================


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    entries: list[Path] = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


source_entries = _source_path_entries(BUNDLE_DIR)
if not source_entries:
    raise RuntimeError(f"No bundled source roots found under {BUNDLE_DIR / 'src'}.")
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"nine18.exact: wrote {pth_path} ({len(source_entries)} source roots)")

# Keep retries for transient setup failures, but never replace setup with notebook-local logic.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    result = None
    for attempt in range(1, 4):
        print(f"nine18.exact: setup command (attempt {attempt}/3): {command}", flush=True)
        result = subprocess.run(command, shell=True, cwd=WORKING_DIR, env=env)
        if result.returncode == 0:
            break
        print(f"nine18.exact: setup command failed with exit {result.returncode}; retrying", flush=True)
        time.sleep(10 * attempt)
    else:
        raise subprocess.CalledProcessError(result.returncode, command)
    env = _command_env()
    os.environ.update(env)

for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# =============================================================================
# 5. Load the exact bundled benchmark and validate the exact agent objects
# =============================================================================

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


def _safe_class_info(obj) -> dict[str, str]:
    cls = obj if inspect.isclass(obj) else obj.__class__
    try:
        file_path = inspect.getfile(cls)
    except Exception:
        file_path = "<unknown>"
    return {
        "class": getattr(cls, "__qualname__", repr(cls)),
        "module": getattr(cls, "__module__", "<unknown>"),
        "file": file_path,
    }


def _is_publicish_attr(name: str) -> bool:
    return not (name.startswith("__") and name.endswith("__"))


def _collect_agent_like_components(root, *, max_depth: int = 5, max_items: int = 400) -> list[dict[str, str]]:
    keywords = ("agent", "solver", "policy", "planner", "controller", "strategy", "swarm", "ensemble", "player")
    seen: set[int] = set()
    stack: list[tuple[str, object, int]] = [("benchmark", root, 0)]
    collected: list[dict[str, str]] = []

    while stack and len(seen) < max_items:
        path, obj, depth = stack.pop()
        obj_id = id(obj)
        if obj_id in seen:
            continue
        seen.add(obj_id)

        info = _safe_class_info(obj)
        text = " ".join([path, info["class"], info["module"], info["file"]]).lower()
        if any(keyword in text for keyword in keywords):
            collected.append({"path": path, **info, "repr": repr(obj)[:500]})

        if depth >= max_depth:
            continue
        if isinstance(obj, dict):
            for key, value in list(obj.items())[:50]:
                stack.append((f"{path}.{key!r}", value, depth + 1))
        elif isinstance(obj, (list, tuple, set, frozenset)):
            for index, value in enumerate(list(obj)[:50]):
                stack.append((f"{path}[{index}]", value, depth + 1))
        elif hasattr(obj, "__dict__"):
            for name, value in list(vars(obj).items())[:80]:
                if _is_publicish_attr(name):
                    stack.append((f"{path}.{name}", value, depth + 1))

    return collected


def _validate_exact_agents(benchmark) -> list[dict[str, str]]:
    solver = getattr(benchmark, "solver", None)
    if solver is None:
        raise RuntimeError("Loaded benchmark has no .solver; refusing to run without the exact bundled solver.")

    components = _collect_agent_like_components(benchmark)
    solver_info = _safe_class_info(solver)
    if not any(item.get("path") == "benchmark.solver" for item in components):
        components.insert(0, {"path": "benchmark.solver", **solver_info, "repr": repr(solver)[:500]})

    suspicious_tokens = ("dummy", "stub", "mock", "fallback", "offline_validation", "placeholder", "toy")
    suspicious = []
    for item in components:
        haystack = " ".join(str(item.get(key, "")) for key in ("path", "class", "module", "file", "repr")).lower()
        if any(token in haystack for token in suspicious_tokens):
            suspicious.append(item)
    if suspicious:
        raise RuntimeError(
            "Refusing to run: agent-like placeholder/fallback components were detected: "
            + json.dumps(suspicious[:10], indent=2)
        )

    manifest = {
        "true_submission": TRUE_SUBMISSION,
        "require_true_submission": REQUIRE_TRUE_SUBMISSION,
        "bundle_dir": str(BUNDLE_DIR),
        "source_entries": [str(entry) for entry in source_entries],
        "target": _safe_class_info(target),
        "benchmark": _safe_class_info(benchmark),
        "solver": solver_info,
        "agent_like_components": components,
    }
    (WORKING_DIR / "exact_agent_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    print("nine18.exact: exact agent manifest written to /kaggle/working/exact_agent_manifest.json")
    print(f"nine18.exact: solver = {solver_info['module']}.{solver_info['class']} from {solver_info['file']}")
    print(f"nine18.exact: agent-like component count = {len(components)}")
    return components


_validate_exact_agents(bm)

# =============================================================================
# 6. Build the official competition game list only
# =============================================================================


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if not TRUE_SUBMISSION:
    raise RuntimeError(
        "Actual scored mode requires the live Kaggle gateway. Refusing to run offline environments."
    )

os.environ.setdefault("ARC_API_KEY", "test-key-123")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
_wait_for_gateway(os.environ["ARC_BASE_URL"])
bm.games = _competition_games()

# Preserve official benchmark semantics: one pass, no custom game weights, no game reordering.
bm.n_passes = 1
bm.game_weights = None

# =============================================================================
# 7. Run the benchmark and let the exact bundle teardown write submission.parquet
# =============================================================================

KAGGLE_HARD_LIMIT_S = 9 * 3600
budget = float(getattr(target, "max_runtime_s", 0.0) or KAGGLE_HARD_LIMIT_S)
# Leave enough time for the official teardown/submission writer. This does not create fake rows.
margin = min(900.0, max(300.0, budget * 0.08))
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=max(1.0, budget - margin))
print(f"nine18.exact: soft_end={soft_end} budget={budget}s margin={margin}s")

result = None
try:
    result = await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    summary = {
        "true_submission": TRUE_SUBMISSION,
        "result_repr": repr(result)[:4000],
        "games": [getattr(game, "env_name", repr(game)) for game in getattr(bm, "games", [])],
        "n_passes": bm.n_passes,
        "game_weights": bm.game_weights,
        "solver": _safe_class_info(getattr(bm, "solver", None)),
    }
    (WORKING_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
    print("nine18.exact: wrote run_summary.json")
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"nine18.exact: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

submission_path = WORKING_DIR / "submission.parquet"
if not submission_path.exists():
    raise RuntimeError(
        "The exact bundled teardown did not create /kaggle/working/submission.parquet. "
        "No fallback parquet was created because this notebook is configured for an actual scored run."
    )
print(f"nine18.exact: real submission exists at {submission_path} ({submission_path.stat().st_size} bytes)")

# =============================================================================
# 8. Show diagnostics when the bundle produced them
# =============================================================================

from html import escape
from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html - minimal diagnostics normally suppresses it in a real submission.")
